# **Modelling and Evaluation Notebook**

## Objectives

* Load the feature-engineered train, test, and inherited houses datasets.
* Train regression models to predict `SalePrice`.
* Compare candidate regression models.
* Tune the selected model.
* Evaluate the final model using R², MAE, RMSE, and actual-vs-predicted plots.
* Save the fitted model and evaluation artifacts for use in the Streamlit dashboard.

## Inputs

* `outputs/datasets/featured/TrainSetFeatured.csv`
* `outputs/datasets/featured/TestSetFeatured.csv`
* `outputs/datasets/featured/InheritedHousesFeatured.csv`
* `outputs/ml_pipeline/predict_sale_price/v1/selected_features.pkl`

## Outputs

* `outputs/ml_pipeline/predict_sale_price/v1/regression_pipeline.pkl`
* `outputs/ml_pipeline/predict_sale_price/v1/model_performance.pkl`
* `outputs/ml_pipeline/predict_sale_price/v1/feature_importance.csv`
* `outputs/ml_pipeline/predict_sale_price/v1/feature_importance.png`
* `outputs/ml_pipeline/predict_sale_price/v1/actual_vs_predicted_train.png`
* `outputs/ml_pipeline/predict_sale_price/v1/actual_vs_predicted_test.png`
* `outputs/ml_pipeline/predict_sale_price/v1/inherited_houses_predictions.csv`

## Additional Comments

* This notebook focuses on model training and evaluation.
* The model is considered successful if it achieves an R² score of at least 0.75 on the test set.

---

# Set Working Directory

Ensure the working directory is set to the project root directory.

In [ ]:
import os
from pathlib import Path

current_dir = Path.cwd()

if current_dir.name == "jupyter_notebooks":
    os.chdir(current_dir.parent)
    print("You set a new current directory")
else:
    print("Current directory is already the project root or another non-notebook directory")

Path.cwd()

---

# Import Packages

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

---

# Load Feature-Engineered Datasets

In [ ]:
TrainSet = pd.read_csv("outputs/datasets/featured/TrainSetFeatured.csv")
TestSet = pd.read_csv("outputs/datasets/featured/TestSetFeatured.csv")
InheritedHouses = pd.read_csv("outputs/datasets/featured/InheritedHousesFeatured.csv")

selected_features = joblib.load("outputs/ml_pipeline/predict_sale_price/v1/selected_features.pkl")

In [ ]:
TrainSet.head()

In [ ]:
TrainSet.shape, TestSet.shape, InheritedHouses.shape

In [ ]:
selected_features

# Split Features and Target

In [ ]:
target = "SalePrice"

X_train = TrainSet[selected_features]
y_train = TrainSet[target]

X_test = TestSet[selected_features]
y_test = TestSet[target]

X_inherited = InheritedHouses[selected_features]

In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape, X_inherited.shape

Validation:

In [ ]:
print("Train missing values:", X_train.isna().sum().sum())
print("Test missing values:", X_test.isna().sum().sum())
print("Inherited missing values:", X_inherited.isna().sum().sum())

In [ ]:
X_train.select_dtypes(include=["object"]).columns.to_list()

Which returns `[]`, as expected.

# Evaluation Functions

The model is evaluated using R², MAE, RMSE, and actual-vs-predicted plots.

R² is the main success metric because the business case defines model success as achieving at least 0.75 R² on the test set. MAE and RMSE are also shown because they express the prediction error in sale-price units.

In [ ]:
def regression_performance(model, X_train, y_train, X_test, y_test):
    """
    Calculate regression performance metrics for train and test sets.
    """
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    performance = pd.DataFrame({
        "Set": ["Train", "Test"],
        "R2": [
            r2_score(y_train, y_train_pred),
            r2_score(y_test, y_test_pred)
        ],
        "MAE": [
            mean_absolute_error(y_train, y_train_pred),
            mean_absolute_error(y_test, y_test_pred)
        ],
        "RMSE": [
            mean_squared_error(y_train, y_train_pred, squared=False),
            mean_squared_error(y_test, y_test_pred, squared=False)
        ]
    })

    return performance.round(3)

In [ ]:
def plot_actual_vs_predicted(y_true, y_pred, title, save_path=None):
    """
    Plot actual sale prices against predicted sale prices.
    """
    plt.figure(figsize=(8, 8))
    sns.scatterplot(x=y_true, y=y_pred)

    min_value = min(y_true.min(), y_pred.min())
    max_value = max(y_true.max(), y_pred.max())

    plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")
    plt.title(title)
    plt.xlabel("Actual Sale Price")
    plt.ylabel("Predicted Sale Price")

    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight")

    plt.show()

# Baseline Model Comparison

In [ ]:
candidate_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=0, max_depth=6),
    "Random Forest": RandomForestRegressor(random_state=0, n_estimators=200),
    "Extra Trees": ExtraTreesRegressor(random_state=0, n_estimators=200),
    "Gradient Boosting": GradientBoostingRegressor(random_state=0)
}

The candidate models are compared using train and test R². The selected model should perform well on the test set without showing excessive overfitting.

In [ ]:
model_comparison = []

for model_name, model in candidate_models.items():
    pipeline = Pipeline([
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    model_comparison.append({
        "Model": model_name,
        "Train R2": r2_score(y_train, y_train_pred),
        "Test R2": r2_score(y_test, y_test_pred),
        "Train MAE": mean_absolute_error(y_train, y_train_pred),
        "Test MAE": mean_absolute_error(y_test, y_test_pred),
        "Train RMSE": mean_squared_error(y_train, y_train_pred, squared=False),
        "Test RMSE": mean_squared_error(y_test, y_test_pred, squared=False)
    })

model_comparison_df = pd.DataFrame(model_comparison).sort_values(
    by="Test R2",
    ascending=False
)

model_comparison_df.round(3)

The baseline model comparison showed that Gradient Boosting achieved the strongest test-set R² score. It also showed less overfitting than the tree ensemble models that produced near-perfect training scores.

For that reason, `GradientBoostingRegressor` was selected for final tuning.

# Hyperparameter Tuning

The best-performing candidate model is tuned using `GridSearchCV`.

The grid is intentionally modest. The aim is to improve the model without creating unnecessary complexity or excessive runtime. The tuning process uses cross-validation on the training set only.

In [ ]:
final_pipeline = Pipeline([
    ("model", GradientBoostingRegressor(random_state=0))
])

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [2, 3],
    "model__min_samples_leaf": [1, 3],
    "model__subsample": [0.8, 1.0]
}

In [ ]:
grid_search = GridSearchCV(
    estimator=final_pipeline,
    param_grid=param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

In [ ]:
grid_search.best_params_

In [ ]:
grid_search.best_score_

In [ ]:
final_model = grid_search.best_estimator_
final_model

# Final Model Performance

The final model is evaluated on the train and test sets.

The project success threshold is R² >= 0.75 on the test set. The test score is the most important metric because it estimates how well the model generalises to unseen data.

In [ ]:
model_performance = regression_performance(
    final_model,
    X_train,
    y_train,
    X_test,
    y_test
)

model_performance

In [ ]:
train_r2 = model_performance.loc[model_performance["Set"] == "Train", "R2"].iloc[0]
test_r2 = model_performance.loc[model_performance["Set"] == "Test", "R2"].iloc[0]

train_r2, test_r2

In [ ]:
SUCCESS_THRESHOLD = 0.75

if test_r2 >= SUCCESS_THRESHOLD:
    print(f"The model meets the business requirement. Test R² = {test_r2:.3f}.")
else:
    print(f"The model does not meet the business requirement. Test R² = {test_r2:.3f}.")

# Actual vs Predicted Plots

In [ ]:
pipeline_dir = Path("outputs/ml_pipeline/predict_sale_price/v1")
pipeline_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

In [ ]:
plot_actual_vs_predicted(
    y_train,
    y_train_pred,
    title="Actual vs Predicted Sale Price - Train Set",
    save_path=pipeline_dir / "actual_vs_predicted_train.png"
)

In [ ]:
plot_actual_vs_predicted(
    y_test,
    y_test_pred,
    title="Actual vs Predicted Sale Price - Test Set",
    save_path=pipeline_dir / "actual_vs_predicted_test.png"
)

The diagonal dashed line represents perfect predictions. Points closer to the line indicate more accurate predictions. Wider spread away from the line indicates larger prediction errors.

# Feature Importance

Feature importance shows which variables contributed most to the model's predictions.

This does not prove that these features cause sale price changes, but it helps explain which features the model found most useful when making predictions.

In [ ]:
model = final_model.named_steps["model"]

feature_importance_df = pd.DataFrame({
    "Feature": selected_features,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance_df.head(15)

In [ ]:
feature_importance_df.to_csv(pipeline_dir / "feature_importance.csv", index=False)

In [ ]:
plt.figure(figsize=(10, 8))
sns.barplot(
    data=feature_importance_df.head(15),
    x="Importance",
    y="Feature"
)
plt.title("Top 15 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.savefig(pipeline_dir / "feature_importance.png", bbox_inches="tight")
plt.show()

# Predict Inherited Houses

The fitted model is used to predict sale prices for Lydia's four inherited houses. These predictions will be displayed in the Streamlit dashboard.

In [ ]:
InheritedHousesDisplay = pd.read_csv(
    "outputs/datasets/cleaned/InheritedHousesCleaned.csv",
    keep_default_na=False
)

inherited_predictions = final_model.predict(X_inherited)

inherited_predictions_df = InheritedHousesDisplay.copy()
inherited_predictions_df["PredictedSalePrice"] = inherited_predictions.round(2)

inherited_predictions_df

In [ ]:
inherited_predictions_df.to_csv(
    pipeline_dir / "inherited_houses_predictions.csv",
    index=False
)

In [ ]:
total_predicted_value = inherited_predictions_df["PredictedSalePrice"].sum()
total_predicted_value

# Save Model

In [ ]:
joblib.dump(final_model, pipeline_dir / "regression_pipeline.pkl")
joblib.dump(model_performance, pipeline_dir / "model_performance.pkl")
joblib.dump(model_comparison_df, pipeline_dir / "model_comparison.pkl")
joblib.dump(grid_search.best_params_, pipeline_dir / "best_model_params.pkl")

In [ ]:
loaded_model = joblib.load(pipeline_dir / "regression_pipeline.pkl")

loaded_test_predictions = loaded_model.predict(X_test)

r2_score(y_test, loaded_test_predictions)

The saved model is reloaded and tested to confirm that it can be used after being saved. This is important because the Streamlit dashboard will load the saved model file to make predictions.

# Conclusions and Next Steps

This notebook trained and evaluated a regression model to predict house sale price.

The main modelling steps were:

* loaded feature-engineered train, test, and inherited houses datasets;
* compared several regression models;
* selected and tuned the best-performing model;
* evaluated the final model using R², MAE, RMSE, and actual-vs-predicted plots;
* generated feature importance information;
* predicted sale prices for Lydia's inherited houses;
* saved the fitted model and evaluation artifacts.

The final Gradient Boosting model achieved a test set R² score of `0.880`, which meets the project success threshold of `0.75`.

The training score is higher than the test score, which is expected, but the test score remains strong enough to indicate that the model generalises reasonably well to unseen data.

Therefore, the model is considered successful for the predictive business requirement.